<a href="https://colab.research.google.com/github/EnzoFerroni/projeto-aplicado-iv-energia-solar/blob/main/notebooks/projeto_aplicado_IV_doc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/scalabrinig/cdProjetoAplicadoIV/d093146488f56dfcf0ef286bcee8efe0e71b9c76/figuras/mackenzie_logo.jpg" width="25%" align="right"/>

# **PROJETO APLICADO IV - Ciência de Dados EaD - 2026/02**

# **Documentação do Projeto**

---

# **Análise e Previsão do Crescimento da Energia Solar Distribuída no Brasil**

---

**Etapa 2 - Referencial Teórico e Cronograma**

São Paulo - SP, 28 de setembro de 2026

Repositório: https://github.com/EnzoFerroni/projeto-aplicado-iv-energia-solar

In [ ]:
#@title **Identificação do Grupo**

#@markdown Integrantes do Grupo, nome completo em ordem alfabética (*informe: \<nome\>, \<matrícula\>*)
Aluno1 = 'Daniel dos Santos da Silva, 10720767' #@param {type:"string"}
Aluno2 = 'Enzo Ferroni, 10417100' #@param {type:"string"}
Aluno3 = 'Vinicius de Souza Sabia, 10721475' #@param {type:"string"}
Aluno4 = '' #@param {type:"string"}
Aluno5 = '' #@param {type:"string"}

# **Resumo**

A expansão da micro e minigeração distribuída (MMGD) solar no Brasil é acompanhada de forma predominantemente descritiva e retrospectiva, e as projeções setoriais disponíveis são construídas em escala nacional e por cenários regulatórios, sem previsões desagregadas por unidade federativa. Este projeto propõe um produto analítico de séries temporais para caracterizar e prever o crescimento da geração solar fotovoltaica distribuída no país, tendo como variável-alvo a potência instalada acumulada (kW) e o número de novas conexões, agregados em periodicidade mensal a partir da data de conexão. A base empregada é o registro administrativo de empreendimentos de MMGD da ANEEL, de cobertura nacional e com identificação por município, complementado pelo Painel de Dados de MMGD da EPE para validação cruzada dos agregados. Metodologicamente, o trabalho compreende a construção das séries mensais, a análise exploratória de tendência, sazonalidade e quebras estruturais associadas à Lei nº 14.300/2022, e a comparação de modelos de previsão — suavização exponencial e ARIMA/SARIMA como referência estatística clássica, Prophet pela robustez a mudanças de tendência e modelos baseados em árvores como alternativa —, avaliados por validação em janelas temporais com as métricas MAE, RMSE e MAPE.

*(Os principais resultados serão incorporados a este resumo na Etapa 4.)*

# **Introdução**

## Contexto temático e área de conhecimento

O presente projeto insere-se na área de **Ciência de Dados**, com ênfase em **análise de séries temporais** aplicada ao setor elétrico brasileiro. As competências mobilizadas envolvem obtenção e preparação de dados públicos, análise exploratória, modelagem estatística e de aprendizado de máquina para previsão, e comunicação de resultados por meio de visualizações. O domínio de aplicação é o da **micro e minigeração distribuída (MMGD)** de energia elétrica, com recorte na fonte **solar fotovoltaica**.

A MMGD é a modalidade em que o próprio consumidor gera parte da energia que consome e injeta o excedente na rede da distribuidora, recebendo créditos de compensação. No Brasil, a modalidade foi regulamentada pela Resolução Normativa ANEEL nº 482/2012 e teve seu marco legal consolidado pela Lei nº 14.300/2022, que definiu as regras de transição do sistema de compensação e classificou como microgeração as centrais de até 75 kW e como minigeração aquelas acima de 75 kW e até 3 MW (BRASIL, 2022; ANEEL, 2026a).

Desde então, a MMGD tornou-se o segmento de maior expansão da matriz elétrica nacional, com forte predomínio da fonte solar fotovoltaica, o que a conecta diretamente a temas de infraestrutura energética, transição para fontes limpas, desenvolvimento econômico regional e sustentabilidade urbana. Nesse sentido, o tema é aderente aos três Objetivos de Desenvolvimento Sustentável adotados no semestre. Ao **ODS 8 - trabalho decente e crescimento econômico**, pela cadeia produtiva e pelos investimentos mobilizados pelo setor solar. Ao **ODS 11 - cidades e comunidades sustentáveis**, pela descentralização da geração e pela redução da pressão sobre a infraestrutura de transmissão. E ao **ODS 16 - paz, justiça e instituições eficazes**, especificamente por suas metas **16.6** (instituições eficazes, responsáveis e transparentes) e **16.10** (acesso público à informação): o projeto apoia-se integralmente em dados abertos publicados pelo regulador e converte esse acervo em evidência sobre os efeitos da Lei nº 14.300/2022, ampliando a capacidade de escrutínio público sobre uma política setorial (ONU, 2026).

## Caracterização do problema

Apesar do volume expressivo de dados públicos disponíveis, o acompanhamento da expansão da geração solar distribuída no Brasil é feito, em grande parte, de forma descritiva e retrospectiva: os painéis oficiais informam o que já ocorreu, mas oferecem pouca capacidade de antecipação em nível desagregado. As projeções setoriais existentes, como as publicadas pela Empresa de Pesquisa Energética, são construídas em escala nacional e por cenários regulatórios, e não fornecem previsões por unidade federativa ou por município (EPE, 2025).

Soma-se a isso o fato de que a adoção da tecnologia é bastante desigual no território nacional. Existem estados e municípios com adesão consolidada e outros com participação marginal, e essa heterogeneidade não é explicada apenas pelo porte populacional. Sem uma leitura temporal desagregada, torna-se difícil identificar quais localidades estão acelerando, quais estão saturando e quais permanecem estagnadas.

> **Problema:** como modelar a evolução histórica da potência instalada e do número de empreendimentos de geração solar distribuída no Brasil, de modo a produzir previsões de curto e médio prazo em nível nacional e estadual, e a caracterizar as diferenças regionais desse crescimento?

Trata-se de um problema típico de séries temporais. A variável de interesse é a **potência instalada acumulada (kW)** e o **número de novas conexões solares fotovoltaicas**, agregados em **periodicidade mensal** a partir da data de conexão de cada empreendimento. A série apresenta tendência de crescimento acentuada, possível sazonalidade intra-anual e, sobretudo, **quebras estruturais** associadas às mudanças regulatórias, com destaque para a corrida por conexões observada às vésperas da vigência das novas regras de compensação instituídas pela Lei nº 14.300/2022 (BRASIL, 2022).

## Motivação

A motivação do grupo decorre da convergência entre a relevância do tema e a qualidade dos dados disponíveis. A energia solar distribuída é hoje um dos vetores mais dinâmicos do setor elétrico brasileiro e está diretamente associada a três questões sensíveis para a sociedade: o custo da energia elétrica para o consumidor, a descarbonização da matriz e a necessidade de planejamento da infraestrutura de distribuição.

A **lacuna** percebida é de natureza analítica. Há dados abertos abundantes, atualizados diariamente e com granularidade municipal, mas há pouca oferta de instrumentos públicos, reprodutíveis e de baixo custo que transformem esse histórico em previsão desagregada. As projeções oficiais são agregadas e dependem de premissas regulatórias e de simulações proprietárias (EPE, 2025), o que abre espaço para uma abordagem alternativa, orientada a dados e replicável em notebook.

Há ainda uma motivação metodológica. A série da MMGD reúne, em um único conjunto real, os fenômenos que caracterizam o estudo de séries temporais: tendência forte, sazonalidade, mudanças de regime e efeitos de antecipação induzidos por regulação. Isso torna o problema adequado para exercitar e comparar diferentes famílias de modelos, e não apenas aplicar um método isolado.

## Objetivo

**Objetivo geral:** desenvolver um produto analítico, em formato de notebook executável, capaz de caracterizar e prever o crescimento da geração solar fotovoltaica distribuída no Brasil, a partir dos dados públicos de micro e minigeração distribuída da ANEEL, com previsões em nível nacional e estadual.

**Objetivos específicos:**

1. Obter, tratar e consolidar a base de empreendimentos de MMGD da ANEEL, filtrando a fonte solar fotovoltaica e organizando os registros em séries temporais mensais por unidade federativa e por município.
2. Realizar a análise exploratória da série, identificando tendência, sazonalidade, mudanças de regime e possíveis quebras estruturais associadas à Lei nº 14.300/2022.
3. Avaliar a qualidade dos dados, tratando registros faltantes, duplicidades e a descontinuidade de atualização ocorrida durante a migração de sistemas da ANEEL em 2025.
4. Implementar e comparar ao menos dois modelos de previsão, contemplando abordagens estatísticas clássicas (ARIMA, suavização exponencial) e alternativas (Prophet ou modelos baseados em árvores).
5. Avaliar o desempenho dos modelos por meio de métricas de erro (MAE, RMSE, MAPE), com validação em janelas temporais.
6. Produzir visualizações que comuniquem a evolução histórica, as previsões e as diferenças regionais de adoção.
7. Disponibilizar todo o material do projeto em repositório público no GitHub, garantindo a reprodutibilidade da solução.

## Justificativa

Do ponto de vista das **contribuições para a sociedade**, o produto analítico pode apoiar decisões de diferentes agentes. Para gestores públicos estaduais e municipais, previsões desagregadas ajudam a dimensionar políticas de incentivo e a identificar territórios com baixa adesão. Para distribuidoras de energia, a antecipação do ritmo de conexões é insumo para planejamento da rede. Para empresas do setor e para o consumidor, a leitura comparativa entre localidades oferece um panorama objetivo do mercado.

Do ponto de vista **científico e técnico**, a contribuição está na aplicação e na comparação sistemática de modelos de séries temporais sobre uma base pública, oficial e de grande volume, com tratamento explícito das quebras estruturais decorrentes da mudança regulatória. O uso de fontes primárias e de código executável favorece a reprodutibilidade.

Quanto à **aplicabilidade**, a solução é viável dentro do prazo e dos recursos do componente: os dados são gratuitos, documentados e acessíveis por download direto; as técnicas previstas estão implementadas em bibliotecas maduras de código aberto; e o produto final é um notebook executável, que pode ser reexecutado sempre que a ANEEL atualizar a base.

# **Descrição da Base de Dados**

## Fonte e forma de coleta

A base principal do projeto é a **Relação de empreendimentos de Mini e Micro Geração Distribuída**, publicada pela Agência Nacional de Energia Elétrica no Portal de Dados Abertos da ANEEL (ANEEL, 2026b). Trata-se de uma **fonte primária e oficial**: os registros são gerados administrativamente, a partir das informações que as distribuidoras enviam ao regulador no momento em que uma unidade consumidora com geração distribuída é conectada à rede. Não há coleta amostral ou por pesquisa de campo, mas **registro censitário** de todas as conexões homologadas no país.

O acesso é público e não requer autenticação. O conjunto é disponibilizado em arquivos **CSV compactados em ZIP** e também em formato **Apache Parquet**, acompanhado de um **dicionário de dados em PDF**. Além do arquivo principal de empreendimentos, o portal disponibiliza arquivos complementares com informações técnicas por fonte de geração, incluindo o arquivo específico da fonte fotovoltaica, que pode ser associado ao arquivo principal pelo código do empreendimento.

Como **fonte complementar**, será utilizado o Painel de Dados de Micro e Minigeração Distribuída (PDGD) da Empresa de Pesquisa Energética (EPE, 2026), para validação cruzada dos agregados e comparação com as projeções oficiais do Plano Decenal de Expansão de Energia (EPE, 2025).

## Informações disponíveis, estrutura e organização

Os dados estão organizados em **formato tabular**, no qual cada linha corresponde a um empreendimento de geração distribuída conectado à rede. A cobertura é **nacional**, com identificação da unidade federativa e do município.

| Tipo de informação | Conteúdo | Uso previsto no projeto |
|---|---|---|
| Temporal | Data de conexão do empreendimento à rede | Eixo temporal da série; base para agregação mensal |
| Técnica | Fonte de geração, potência instalada (kW) e modalidade | Filtro da fonte solar e variável-alvo da previsão |
| Geográfica | Unidade federativa, município e código IBGE | Desagregação regional e comparação entre estados |
| Setorial | Distribuidora, classe e subgrupo de consumo | Segmentação por perfil de adotante |
| Identificação | Código do empreendimento | Junção com o arquivo técnico da fonte fotovoltaica |

*Fonte: elaborado pelos autores com base em ANEEL (2026b).*

A base, em sua forma original, **não constitui uma série temporal**: ela é um cadastro de eventos. A construção da série é, portanto, uma etapa de pré-processamento do projeto e consistirá em filtrar os registros da fonte solar fotovoltaica, agrupá-los pela data de conexão em períodos mensais e somar a potência instalada e a contagem de empreendimentos por período e por localidade, obtendo tanto a série de novas conexões quanto a série de capacidade acumulada.

## Período de coleta, atualização e limitações

O conjunto reúne registros **a partir de dezembro de 2008** e é **atualizado diariamente**, conforme os metadados do portal (ANEEL, 2026b).

Três limitações já foram identificadas e serão tratadas explicitamente no trabalho:

1. **Quebras estruturais de origem regulatória**, especialmente a antecipação de conexões ocorrida no período de transição da Lei nº 14.300/2022.
2. **Interrupção temporária na atualização** da base entre 23/09/2025 e 13/11/2025, decorrente da migração do sistema SISGD para o novo sistema de MMGD da ANEEL, que pode ter produzido represamento e posterior acúmulo de registros.
3. **Defasagem entre a data de conexão e a data de registro** no cadastro, que recomenda cautela na interpretação da ponta final da série.

### Síntese da base de dados

| Item | Descrição |
|---|---|
| Fonte principal | ANEEL - Portal de Dados Abertos (fonte primária e oficial) |
| Conjunto de dados | Relação de empreendimentos de Mini e Micro Geração Distribuída |
| Formatos | CSV compactado em ZIP e Apache Parquet, com dicionário de dados em PDF |
| Granularidade | Um registro por empreendimento conectado à rede |
| Cobertura geográfica | Nacional, com identificação por UF e município |
| Período de coleta | Registros a partir de dezembro de 2008 |
| Atualização | Diária, conforme metadados do portal |
| Forma de coleta | Registro administrativo enviado pelas distribuidoras ao regulador |
| Fonte complementar | EPE - Painel PDGD e Caderno de MMGD do PDE 2035 |
| Pré-processamento | Filtro da fonte solar, agregação mensal por data de conexão e soma da potência por localidade |

*Fonte: elaborado pelos autores com base em ANEEL (2026b) e EPE (2025, 2026).*

# **Solução Proposta**

O produto analítico será desenvolvido em Python, em formato de notebook executável e versionado neste repositório, contemplando as etapas de obtenção, preparação, análise exploratória, modelagem e avaliação dos resultados.

**1. Análise exploratória.** Visualizações da evolução mensal da potência instalada e do número de conexões, decomposição da série em tendência, sazonalidade e resíduo, e indicadores comparativos de crescimento por unidade federativa.

**2. Modelagem.** Serão implementados e comparados modelos de famílias distintas: métodos estatísticos clássicos, como suavização exponencial e ARIMA, tomados como referência metodológica consolidada (HYNDMAN; ATHANASOPOULOS, 2021), e abordagens mais robustas a mudanças de tendência, como o Prophet, que modela a série de forma aditiva por meio de componentes de tendência, sazonalidade e pontos de mudança (TAYLOR; LETHAM, 2018). Modelos baseados em árvores de decisão poderão ser avaliados como alternativa, mediante engenharia de atributos temporais.

**3. Avaliação.** Validação em janelas temporais, com separação cronológica entre treino e teste, comparando os desempenhos por métricas de erro de previsão. O critério de escolha do modelo final será a combinação entre acurácia preditiva e capacidade de lidar com as quebras estruturais identificadas.

### Aderência da proposta aos critérios da etapa

| Critério | Como o projeto atende |
|---|---|
| Atualidade e importância | A MMGD é o segmento de maior expansão da matriz elétrica brasileira, ligada à transição energética e ao custo da energia. |
| Fonte dos dados | Base pública, oficial, primária, de cobertura nacional, atualizada diariamente e com dicionário de dados documentado. |
| Adequação a séries temporais | A data de conexão permite construir séries mensais com tendência, sazonalidade e quebras estruturais identificáveis. |
| Solução proposta | Notebook executável e reprodutível, com comparação entre modelos estatísticos e de aprendizado de máquina. |
| Contribuição social | Fornece previsões desagregadas úteis a gestores públicos, distribuidoras e agentes do setor, hoje indisponíveis publicamente. |
| Aderência aos ODS | Relaciona-se aos ODS 8, 11 e 16, definidos para o semestre; o ODS 16 pelas metas 16.6 e 16.10, via uso de dados abertos e transparência regulatória. |

# **Referencial Teórico**

Uma série temporal é um conjunto de observações ordenadas no tempo, em que cada valor costuma depender dos valores anteriores. Neste projeto a série é montada a partir da data de conexão de cada sistema solar e somada mês a mês. Para entender seu comportamento usamos a decomposição, que separa a série em tendência, sazonalidade e resíduo, e as funções de autocorrelação (ACF) e autocorrelação parcial (PACF), que medem o quanto um mês se relaciona com os meses anteriores (HYNDMAN; ATHANASOPOULOS, 2021). Outro conceito central é a estacionariedade, ou seja, média e variância estáveis ao longo do tempo, exigida por modelos como o ARIMA e normalmente obtida com diferenciação ou com a transformação Box-Cox (BOX; COX, 1964). Também é importante a ideia de quebra estrutural, uma mudança brusca de nível ou de ritmo da série, como a corrida por conexões antes da vigência da Lei nº 14.300/2022 (BRASIL, 2022).

Entre os trabalhos correlatos, o mais próximo do nosso é o de Costa et al. (2023), que processaram mais de 1,4 milhão de registros da ANEEL e compararam ARIMA, SARIMA, suavização exponencial de Holt-Winters, o modelo de difusão de Bass e redes neurais para prever a potência instalada no Brasil e em cada região. O Holt-Winters teve o menor erro no total nacional, mas o melhor modelo mudou de região para região, o que mostra que não existe uma técnica única para todos os recortes. Tokuue (2024) usou a mesma base entre 2012 e 2022 e concluiu que o modelo ETS superou o ARIMA, porém com MAPE perto de 23% na validação, um erro alto para quem precisa planejar. O modelo de Bass (BASS, 1969), usado também por Mejdalani, López Soto e Hallack (2018) para simular a adoção residencial, descreve bem a curva em S típica de novas tecnologias, mas depende de hipóteses sobre o tamanho do mercado e responde mal a choques regulatórios. Tavares e Christo (2025) compararam SARIMA, Prophet e redes neurais na geração elétrica mensal do país e o Prophet foi o melhor para a fonte solar, graças à sua tendência por partes com pontos de mudança (TAYLOR; LETHAM, 2018). As redes neurais, por outro lado, pedem séries longas e são difíceis de interpretar, e a série mensal da MMGD tem só cerca de 200 pontos. Modelos de árvores como o LightGBM, base das melhores soluções da competição M5 (MAKRIDAKIS; SPILIOTIS; ASSIMAKOPOULOS, 2022), são uma boa alternativa quando há muitas séries parecidas, como as 27 unidades federativas, mas não extrapolam a tendência sozinhos e precisam de atributos de defasagem e de séries diferenciadas.

Com base nisso o grupo vai usar como referência os modelos clássicos ETS e SARIMA, que funcionaram bem nos estudos com a mesma base, e compará-los com o Prophet, pela forma como lida com mudanças de tendência, e com um modelo de árvores treinado com todas as UFs ao mesmo tempo. O diferencial em relação aos trabalhos citados está na previsão por estado, no uso de dados atualizados até 2026 e no tratamento explícito da quebra ligada à Lei nº 14.300/2022 e da falha de atualização da base em 2025.

# **Diagrama de Solução**

O pipeline da solução tem sete etapas encadeadas, todas executadas no mesmo notebook para que qualquer pessoa consiga reproduzir o resultado do zero (Figura 1).

<img src="https://raw.githubusercontent.com/EnzoFerroni/projeto-aplicado-iv-energia-solar/main/figuras/pipeline_solucao.png" width="100%"/>

*Figura 1. Pipeline da solução proposta. Fonte: elaborado pelos autores.*

**Coleta e limpeza.** O notebook baixa direto do Portal de Dados Abertos da ANEEL o arquivo de empreendimentos em formato Parquet, que é mais leve e mais rápido de ler do que o CSV. Os arquivos brutos ficam em `data/raw` e não vão para o GitHub. Em seguida mantemos só os empreendimentos de fonte solar fotovoltaica, convertemos a data de conexão e a potência para os tipos corretos, removemos registros repetidos pelo código do empreendimento e descartamos datas fora do intervalo esperado e potências nulas ou negativas.

**Construção das séries.** Os registros são agrupados pelo mês de conexão, gerando três séries para o Brasil e para cada UF, que são o número de novas conexões, a potência nova em kW e a potência acumulada. Os meses mais recentes, ainda incompletos por causa da demora no registro, são cortados, e o período de 23/09/2025 a 13/11/2025, em que a base ficou sem atualização, é marcado para análise separada. As séries prontas são salvas em `data/processed`. Nesta etapa também conferimos os totais com o painel PDGD da EPE (EPE, 2026).

**Análise exploratória.** Fazemos gráficos da série nacional e por UF, calculamos ACF e PACF, aplicamos os testes ADF e KPSS para verificar a estacionariedade, decompomos a série com o método STL (CLEVELAND et al., 1990) e medimos a força da tendência e da sazonalidade. Aqui decidimos se é preciso usar transformação logarítmica ou Box-Cox e criamos uma variável indicadora para o período da Lei nº 14.300/2022.

**Modelagem.** Começamos com dois modelos simples de referência, o ingênuo sazonal e o de tendência linear (drift), que funcionam como piso de desempenho. Depois ajustamos o ETS, o SARIMAX com a variável indicadora da lei, o Prophet com pontos de mudança e um LightGBM global com atributos de defasagem, médias móveis e mês do ano.

**Avaliação.** Usamos validação com origem móvel, em que o modelo é treinado até certo mês, prevê os 12 meses seguintes e a janela avança um passo. Comparamos os modelos por MAE, RMSE e MAPE, e usamos o MASE como apoio nas UFs pequenas, onde valores perto de zero distorcem o MAPE. Olhamos também o desempenho perto das quebras. O modelo final é o que tiver menor erro médio sem perder estabilidade entre as UFs.

**Produto.** Por fim geramos as previsões de 12 e 24 meses para o Brasil e para cada UF, com intervalos de previsão, e organizamos gráficos e tabelas no notebook e no README do repositório.

A solução é viável dentro do prazo porque os dados são públicos e gratuitos e todas as etapas usam bibliotecas abertas e maduras do Python (pandas, pyarrow, statsmodels, prophet e lightgbm), que rodam no Google Colab sem custo.

# **Cronograma**

O cronograma cobre todas as atividades do projeto de setembro a novembro de 2026, e não só as datas da disciplina. As tarefas foram divididas entre os integrantes e cada entrega tem alguns dias de folga para revisão. A versão completa, com linha do tempo e divisão de responsabilidades, está em [Cronograma.md](https://github.com/EnzoFerroni/projeto-aplicado-iv-energia-solar/blob/main/Cronograma.md).

### ETAPA 2 – Referencial Teórico e Cronograma (28 dias)

**Segunda entrega: 28 de setembro**

| **Período** | **Atividades** | **Responsável** |
| --- | --- | --- |
| 01/09 - 12/09 | Levantamento bibliográfico de trabalhos correlatos | Todos |
| 08/09 - 18/09 | Escrita do referencial teórico | Vinícius Sabiá |
| 10/09 - 20/09 | Desenho e discussão do pipeline da solução | Enzo Ferroni |
| 14/09 | Checkpoint da Etapa 2 com o professor | Todos |
| 15/09 - 22/09 | Elaboração do cronograma detalhado | Daniel dos Santos |
| 23/09 - 28/09 | Revisão do notebook e publicação no GitHub | Enzo Ferroni |

### ETAPA 3 – Implementação Parcial (28 dias)

**Terceira entrega: 26 de outubro**

| **Período** | **Atividades** | **Responsável** |
| --- | --- | --- |
| 29/09 - 03/10 | Script de coleta automática da base da ANEEL | Enzo Ferroni |
| 01/10 - 08/10 | Limpeza e filtro da fonte solar fotovoltaica | Daniel dos Santos |
| 05/10 | Checkpoint da Etapa 3 com o professor | Todos |
| 06/10 - 11/10 | Construção das séries mensais do Brasil e das UFs | Daniel dos Santos |
| 09/10 - 13/10 | Conferência dos agregados com o painel da EPE | Vinícius Sabiá |
| 12/10 - 18/10 | Análise gráfica da série nacional e por UF | Vinícius Sabiá |
| 12/10 - 19/10 | ACF, PACF, testes de estacionariedade e decomposição STL | Enzo Ferroni |
| 16/10 - 20/10 | Tratamento das quebras (Lei 14.300 e migração do SISGD) | Daniel dos Santos |
| 19/10 - 23/10 | Modelos de referência e primeiro modelo base (ETS) | Enzo Ferroni |
| 23/10 - 26/10 | Atualização do pipeline e do cronograma e entrega da Etapa 3 | Todos |

### ETAPA 4 – Implementação e Entrega Final (35 dias)

**Quarta e última entrega: 30 de novembro**

| **Período** | **Atividades** | **Responsável** |
| --- | --- | --- |
| 27/10 - 04/11 | Modelo SARIMAX com variável da Lei 14.300 | Daniel dos Santos |
| 27/10 - 04/11 | Modelo Prophet com pontos de mudança | Vinícius Sabiá |
| 29/10 - 07/11 | Modelo LightGBM global para as UFs | Enzo Ferroni |
| 09/11 | Checkpoint da Etapa 4 com o professor | Todos |
| 09/11 - 15/11 | Validação com origem móvel e comparação dos modelos | Todos |
| 14/11 - 18/11 | Previsões finais do Brasil e das UFs com intervalos | Enzo Ferroni |
| 16/11 - 22/11 | Resultados, discussão e conclusão no notebook | Daniel dos Santos e Vinícius Sabiá |
| 20/11 - 26/11 | Slides e gravação do vídeo de apresentação | Todos |
| 24/11 - 29/11 | Revisão final, README e teste de execução do zero no Colab | Enzo Ferroni |
| 30/11 | Entrega final do projeto | Todos |

<img src="https://raw.githubusercontent.com/EnzoFerroni/projeto-aplicado-iv-energia-solar/main/figuras/cronograma_gantt.png" width="100%"/>

*Figura 2. Linha do tempo do projeto. Fonte: elaborado pelos autores.*

---
## Seções das próximas etapas

As seções abaixo serão preenchidas nas Etapas 3 e 4 e estão mantidas aqui para preservar a estrutura da documentação do projeto.

# **EDA e Pré-processamento dos dados**

*(Etapa 3)* Exploração, análise e preparação dos dados.

# **Modelos**

*(Etapas 3 e 4)* Criação, refinamento e validação dos modelos propostos.

# **Resultados**

*(Etapa 4)* Métricas, tabelas comparativas e resultados obtidos.

# **Discussão e Conclusão**

*(Etapa 4)* Análise crítica, alcance do objetivo e melhorias futuras.

# **Apresentação**

*(Etapa 4)* Link do vídeo de apresentação.

# **Referências**

AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA. **Micro e minigeração distribuída.** Brasília: ANEEL, 2026a. Disponível em: https://www.gov.br/aneel/pt-br/assuntos/geracao-distribuida. Acesso em: 1 set. 2026.

AGÊNCIA NACIONAL DE ENERGIA ELÉTRICA. **Relação de empreendimentos de mini e micro geração distribuída.** Brasília: Portal de Dados Abertos da ANEEL, 2026b. Disponível em: https://dadosabertos.aneel.gov.br/dataset/relacao-de-empreendimentos-de-geracao-distribuida. Acesso em: 1 set. 2026.

BASS, F. M. A new product growth for model consumer durables. **Management Science,** v. 15, n. 5, p. 215-227, 1969.

BOX, G. E. P.; COX, D. R. An analysis of transformations. **Journal of the Royal Statistical Society: Series B,** v. 26, n. 2, p. 211-252, 1964.

BRASIL. **Lei nº 14.300, de 6 de janeiro de 2022.** Institui o marco legal da microgeração e minigeração distribuída, o Sistema de Compensação de Energia Elétrica (SCEE) e o Programa de Energia Renovável Social (PERS). Brasília, DF: Presidência da República, 2022. Disponível em: https://www.planalto.gov.br/ccivil_03/_ato2019-2022/2022/lei/l14300.htm. Acesso em: 1 set. 2026.

CLEVELAND, R. B.; CLEVELAND, W. S.; McRAE, J. E.; TERPENNING, I. STL: a seasonal-trend decomposition procedure based on loess. **Journal of Official Statistics,** v. 6, n. 1, p. 3-73, 1990.

COSTA, V. B. F.; SCIANNI, L.; MIRANDA, R. C.; BONATTO, B. D. Assessment of the status and trends of photovoltaic distributed generation in Brazil: an in-depth approach based on big data processing. **Solar Energy,** v. 249, p. 694-711, 2023. DOI: 10.1016/j.solener.2022.12.018.

EMPRESA DE PESQUISA ENERGÉTICA. **Micro e minigeração distribuída & baterias atrás do medidor:** Plano Decenal de Expansão de Energia 2035. Rio de Janeiro: EPE, 2025. Disponível em: https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes. Acesso em: 1 set. 2026.

EMPRESA DE PESQUISA ENERGÉTICA. **Painel de dados de micro e minigeração distribuída (PDGD).** Rio de Janeiro: EPE, 2026. Disponível em: https://dashboard.epe.gov.br/apps/pdgd/. Acesso em: 1 set. 2026.

HYNDMAN, R. J.; ATHANASOPOULOS, G. **Forecasting: principles and practice.** 3. ed. Melbourne: OTexts, 2021. Disponível em: https://otexts.com/fpp3/. Acesso em: 1 set. 2026.

MAKRIDAKIS, S.; SPILIOTIS, E.; ASSIMAKOPOULOS, V. M5 accuracy competition: results, findings, and conclusions. **International Journal of Forecasting,** v. 38, n. 4, p. 1346-1364, 2022. DOI: 10.1016/j.ijforecast.2021.11.013.

MEJDALANI, A.; LÓPEZ SOTO, D.; HALLACK, M. **The impact of net metering policy design on the adoption rate of solar photovoltaic systems:** a simulation using calibrated data from Brazil. Washington, DC: Inter-American Development Bank, 2018. (IDB Technical Note, 1593).

ORGANIZAÇÃO DAS NAÇÕES UNIDAS. **Objetivos de desenvolvimento sustentável.** Brasília: ONU Brasil, 2026. Disponível em: https://brasil.un.org/pt-br/sdgs. Acesso em: 1 set. 2026.

TAVARES, H. P.; CHRISTO, E. S. Forecasting Brazil's energy matrix evolution: a multi-method approach to evaluate alignment with SDG 7.2. **Green Energy and Environmental Technology,** 2025. DOI: 10.5772/geet20250064.

TAYLOR, S. J.; LETHAM, B. Forecasting at scale. **The American Statistician,** v. 72, n. 1, p. 37-45, 2018. DOI: 10.1080/00031305.2017.1380080.

TOKUUE, D. A. A. **Análise e aplicação de previsões de séries temporais para o crescimento da geração distribuída de energia fotovoltaica no Brasil.** 2024. Trabalho de Conclusão de Curso (Graduação em Engenharia de Produção), Universidade Federal de Santa Catarina, Florianópolis, 2024. Disponível em: https://repositorio.ufsc.br/handle/123456789/255886. Acesso em: 28 set. 2026.